# 第 5 周 · 周五：CTE 与子查询——让复杂 SQL 可读可维护

> 难度：⭐⭐⭐ | 今日目标：掌握CTE的写法，能用CTE把复杂分析拆成清晰步骤
> 配对课程：01python_learn W5D5（文本向量化基础）
> 前置依赖：周一至周四SQL知识 | 后续用途：数仓ETL、复杂分析SQL
> 扩展阅读：02analyst_thinking/week_01/day_04（CTE与子查询完整知识图谱）
> 面试地位：大厂SQL面试必考，约占SQL考题的15%

---

## 一、核心概念与第一性原理

### 1.1 CTE 的本质——给子查询起名字

CTE(Common Table Expression)的本质是给子查询起名字。它不改变执行效率（大多数数据库优化器会将CTE优化为等价的子查询），但极大地改变了代码的可读性和可维护性。

### 1.2 子查询的痛苦——"子查询地狱"

```sql
SELECT ... FROM (
  SELECT ... FROM (
    SELECT ... FROM orders WHERE ...
  ) t1 WHERE ...
) t2 GROUP BY ...
```

三层嵌套，你从内往外读，读到最外层已经忘了最内层是什么。这就是"子查询地狱"——认知负荷超过人脑工作记忆的容量。

### 1.3 CTE 的救赎——自上而下的逻辑流

```sql
WITH
-- 步骤1：过滤本月订单
monthly_orders AS (
  SELECT * FROM orders WHERE dt >= '2026-01-01' AND dt < '2026-02-01'
),
-- 步骤2：计算每个用户的消费
user_summary AS (
  SELECT user_id, COUNT(*) AS order_cnt, SUM(amount) AS total
  FROM monthly_orders GROUP BY user_id
),
-- 步骤3：分群
user_segments AS (
  SELECT *,
    CASE WHEN total > 1000 THEN '高价值'
         WHEN total > 300 THEN '中等'
         ELSE '低价值' END AS segment
  FROM user_summary
)
-- 步骤4：汇总
SELECT segment, COUNT(*) AS users, AVG(total) AS avg_spend
FROM user_segments GROUP BY segment;
```

从上往下读，每一步做一件事。逻辑清晰，容易调试。这就是CTE的价值——让你的分析思路直接体现在SQL结构上。

### 1.4 CTE vs 子查询——决策框架

| 场景 | 推荐方案 | 原因 |
|------|---------|------|
| 需要多次引用同一子查询 | CTE | 避免重复代码，DRY原则 |
| 超过2层嵌套 | CTE | 嵌套太深难以阅读 |
| 需要递归 | CTE | 只有CTE支持递归 |
| 简单查询（1层） | 子查询 | 更简洁 |
| 需要被多个查询复用 | 临时表 | CTE作用域只有当前查询 |
| 大数据量中间结果 | 临时表 | 可以建索引 |

### 1.5 CTE 的三个核心优势

1. **可读性**：从上到下顺序执行（vs 子查询"从内到外"）
2. **可复用性**：同一个CTE可以被后续多个CTE或主查询引用
3. **可调试性**：每一步都能单独SELECT出来验证，不需要改整个查询

### 1.6 递归CTE——声明式的循环

递归CTE是用声明式SQL实现命令式循环，主要处理层级数据（组织架构、品类树、文件目录）。

```sql
WITH RECURSIVE cte_name AS (
    -- 锚定成员：递归起点
    SELECT ... FROM table WHERE parent_id IS NULL
    UNION ALL
    -- 递归成员：引用CTE自身
    SELECT ... FROM table t JOIN cte_name c ON t.parent_id = c.id
)
SELECT * FROM cte_name;
```

### 1.7 面试高频考点

**面试官常问**：
- "CTE和子查询有什么区别？"
- "什么时候用CTE，什么时候用子查询？"
- "递归CTE的语法是什么？"
- "CTE和临时表有什么区别？"

**回答要点**：
- CTE更可读，子查询更灵活
- 多次引用/超过2层嵌套/需要递归时用CTE
- 递归CTE = 锚定成员 + 递归成员 + UNION ALL
- CTE是逻辑视图（查询级别），临时表是物理存储（会话级别）

> **今天最重要的认知**：CTE不只是一层"语法包装"——它让你的分析思路直接体现在SQL结构上。"先过滤→再聚合→再分群→最后汇总"，每个WITH子句对应你分析逻辑中的一步。一个超过3层嵌套的SQL，用CTE改写后清晰度提升3倍。

---

## 二、知识点详细拆解

### 2.1 CTE语法详解

**单CTE**：
```sql
WITH cte_name AS (
    SELECT column1, column2 FROM table WHERE condition
)
SELECT * FROM cte_name;
```

**多CTE链式定义**（每个CTE可以用前面定义的CTE）：
```sql
WITH
step1 AS (SELECT ... FROM raw_data),
step2 AS (SELECT ... FROM step1 WHERE ...),
step3 AS (SELECT ... FROM step2 GROUP BY ...)
SELECT * FROM step3;
```

**CTE多次引用**：
```sql
WITH user_total AS (
    SELECT user_id, SUM(amount) AS total FROM orders GROUP BY user_id
)
SELECT AVG(total) AS overall_avg,
       (SELECT AVG(total) FROM user_total WHERE total > 500) AS high_avg
FROM user_total;
-- 同一CTE被引用了两次，不需要重复写子查询
```

### 2.2 CTE的物化行为——不同数据库的差异

**为什么重要**：影响性能的关键因素。

| 数据库 | 默认行为 | 控制方式 |
|--------|---------|---------|
| MySQL 8.0+ | 不物化（可能重复执行） | 无法手动控制 |
| PostgreSQL | 不物化 | AS MATERIALIZED / AS NOT MATERIALIZED |
| SQL Server | 不物化 | 优化器自动决定 |
| BigQuery | 自动决定 | 无法手动控制 |

**物化 vs 非物化**：
- 物化：CTE结果存储在内存中，多次引用时只计算一次。适合数据量大但被多次引用的CTE。
- 非物化：CTE像宏一样被展开，每次引用都重新计算。适合数据量小或只引用一次的CTE。

### 2.3 临时表 vs CTE——详细对比

| 特性 | 临时表 | CTE |
|------|--------|-----|
| 存储 | 物理存储（磁盘） | 逻辑视图（内存/计算） |
| 生命周期 | 会话级别 | 查询级别（执行完即释放） |
| 可索引 | ✅ | ❌ |
| 可被多个查询复用 | ✅ | ❌（只在当前查询有效） |
| 大数据量 | 更优 | 一般 |
| 代码简洁度 | 差（需要CREATE/DROP） | 好（WITH一句） |

**选择决策**：
- 中间结果需要在多个查询中使用 → 临时表
- 中间结果只在当前查询中使用 → CTE
- 中间结果数据量很大（>100万行）且被多次引用 → 临时表+索引

### 2.4 递归CTE详解

**为什么重要**：处理层级数据的唯一SQL方案。

**执行过程**：
1. 执行锚定成员，得到初始结果集R0
2. 将R0代入递归成员，得到下一级结果R1
3. 将R1代入递归成员，得到R2
4. 重复直到递归成员返回空集
5. UNION ALL合并所有结果

**应用场景**：
- 组织架构：从CEO找到所有下属
- 品类树：从根品类找到所有子品类
- 评论链：展开所有嵌套回复
- 日期序列：生成连续日期

### 2.5 子查询的位置和用途

| 位置 | 类型 | 用途 | 返回值 |
|------|------|------|--------|
| SELECT | 标量子查询 | 计算单个值 | 单值 |
| FROM | 派生表 | 作为数据源 | 表 |
| WHERE | 过滤子查询 | 筛选行 | 单值或列 |
| HAVING | 组过滤子查询 | 筛选组 | 单值 |

**标量子查询示例**：
```sql
SELECT user_id, amount,
       amount / (SELECT AVG(amount) FROM orders) AS ratio_to_avg
FROM orders;
```

### 2.6 CTE + 窗口函数的协同使用

**为什么重要**：CTE和窗口函数是黄金组合。

```sql
-- 典型模式：CTE做计算，窗口函数做排名
WITH user_metrics AS (
    SELECT user_id,
           SUM(amount) AS total_amount,
           COUNT(*) AS order_cnt,
           MAX(order_date) AS last_order_date
    FROM orders
    GROUP BY user_id
)
SELECT *,
       ROW_NUMBER() OVER(ORDER BY total_amount DESC) AS rank,
       total_amount / SUM(total_amount) OVER() AS pct_of_total
FROM user_metrics;
```

### 2.7 CTE的性能注意事项

- 过多CTE嵌套可能让优化器选择次优方案
- 不同数据库对CTE的优化策略不同，必要时用EXPLAIN验证
- 被多次引用的大数据量CTE可能被重复计算（如果优化器选择不物化）
- 递归CTE有默认最大递归深度限制（MySQL默认1000层）

---

## 三、动手练习

使用Python的sqlite3练习CTE的各种用法。

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

n = 200
df = pd.DataFrame({
    'user_id': np.random.choice(range(1, 31), n),
    'order_date': np.random.choice(pd.date_range('2026-01-01', '2026-01-31'), n),
    'amount': np.round(np.random.lognormal(4, 0.8, n), 2),
    'category': np.random.choice(['服装', '数码', '食品', '美妆'], n)
})
df['order_date'] = df['order_date'].astype(str)
df.to_sql('orders', conn, index=False)

# 组织架构表
pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5, 6, 7],
    'name': ['CEO', 'CTO', 'VP Eng', 'Senior Eng', 'Engineer', 'VP Product', 'PM'],
    'manager_id': [None, 1, 2, 3, 4, 1, 6],
    'dept': ['管理层', '管理层', '技术部', '技术部', '技术部', '产品部', '产品部']
}).to_sql('org', conn, index=False)

print(f"订单数据: {n} 行, {df['user_id'].nunique()} 个用户")
print(f"组织架构: 7人")
print()

### 练习1：CTE基础——将子查询改写为CTE

In [ ]:
print("=" * 60)
print("练习1：CTE vs 子查询——找出高价值用户（消费>500）")

print()
print("旧写法（子查询嵌套）：")
print("SELECT * FROM (")
print("  SELECT user_id, SUM(amount) AS total FROM orders GROUP BY user_id")
print(") WHERE total > 500 ORDER BY total DESC")

print()
print("新写法（CTE）：")
q1 = """WITH user_total AS (
  SELECT user_id, SUM(amount) AS total, COUNT(*) AS cnt
  FROM orders GROUP BY user_id
)
SELECT user_id, total, cnt,
  CASE WHEN total > 500 THEN '高价值' ELSE '普通' END AS label
FROM user_total
WHERE total > 500
ORDER BY total DESC"""
print(pd.read_sql(q1, conn))

print()
print("对比：CTE版本从上往下读，逻辑一目了然")
print("  步骤1：计算每个用户的总消费和订单数")
print("  步骤2：筛选高价值用户并打标签")

### 练习2：多CTE链式操作——从底层到汇总

In [ ]:
print("=" * 60)
print("练习2：多CTE链式操作——用户分层分析")

q2 = """WITH
-- 步骤1：按用户+品类计算消费
user_category AS (
    SELECT user_id, category,
        SUM(amount) AS total,
        COUNT(*) AS order_cnt
    FROM orders
    GROUP BY user_id, category
),
-- 步骤2：在每个品类内排名
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER(PARTITION BY category ORDER BY total DESC) AS rn
    FROM user_category
),
-- 步骤3：筛选Top3
top3 AS (
    SELECT category, user_id, total, order_cnt, rn
    FROM ranked
    WHERE rn <= 3
)
-- 步骤4：按品类汇总
SELECT category,
    COUNT(*) AS top_users,
    ROUND(AVG(total), 1) AS avg_top_amount,
    ROUND(AVG(order_cnt), 1) AS avg_top_orders
FROM top3
GROUP BY category
ORDER BY avg_top_amount DESC"""
print(pd.read_sql(q2, conn))

print()
print("4个CTE，每个CTE都是分析逻辑中的一步：")
print("  1. user_category: 用户×品类粒度的消费统计")
print("  2. ranked: 品类内排名")
print("  3. top3: 取Top3")
print("  4. 最终：按品类汇总Top3的指标")
print("你可以随时SELECT * FROM top3 来验证第3步的结果")

### 练习3：CTE多次引用——找高于平均和低于平均的用户

In [ ]:
print("=" * 60)
print("练习3：CTE多次引用——比平均高的用户 vs 比平均低的用户")

q3 = """WITH user_total AS (
    SELECT user_id, SUM(amount) AS total
    FROM orders
    GROUP BY user_id
),
stats AS (
    SELECT AVG(total) AS avg_total FROM user_total
)
SELECT
    'above_avg' AS group_type,
    COUNT(*) AS user_count,
    ROUND(AVG(ut.total), 1) AS avg_spend
FROM user_total ut, stats s
WHERE ut.total > s.avg_total
UNION ALL
SELECT
    'below_avg' AS group_type,
    COUNT(*) AS user_count,
    ROUND(AVG(ut.total), 1) AS avg_spend
FROM user_total ut, stats s
WHERE ut.total <= s.avg_total"""
print(pd.read_sql(q3, conn))
print()
print("  user_total CTE被引用了2次（above和below各一次）")
print("  stats CTE也被引用了2次")
print("  如果没有CTE，需要写成两个完全相同的子查询——代码重复，难以维护")

### 练习4：递归CTE——查询组织架构

In [ ]:
print("=" * 60)
print("练习4：递归CTE——从CTO找到所有下属")

q4 = """WITH RECURSIVE hierarchy AS (
    -- 锚定成员：从CTO开始
    SELECT emp_id, name, manager_id, dept, 1 AS level
    FROM org
    WHERE name = 'CTO'
    UNION ALL
    -- 递归成员：找下属
    SELECT o.emp_id, o.name, o.manager_id, o.dept, h.level + 1
    FROM org o
    JOIN hierarchy h ON o.manager_id = h.emp_id
)
SELECT * FROM hierarchy ORDER BY level, emp_id"""
print(pd.read_sql(q4, conn))

print()
print("递归CTE的执行过程：")
print("  R0: 找到CTO（level=1）")
print("  R1: 找到CTO的直接下属：VP Eng（level=2）")
print("  R2: 找到VP Eng的下属：Senior Eng（level=3）")
print("  R3: 找到Senior Eng的下属：Engineer（level=4）")
print("  R4: Engineer没有下属 → 递归终止")

### 练习5：挑战练习——用CTE实现复杂的用户行为分析

In [ ]:
print("=" * 60)
print("练习5：复杂分析——用户购买频率×品类偏好分析")

q5 = """WITH
-- 1. 用户基础指标
user_base AS (
    SELECT user_id,
        SUM(amount) AS total_amount,
        COUNT(*) AS order_cnt,
        COUNT(DISTINCT category) AS category_cnt,
        MIN(order_date) AS first_order,
        MAX(order_date) AS last_order
    FROM orders
    GROUP BY user_id
),
-- 2. 用户最爱的品类
user_fav_category AS (
    SELECT user_id, category AS fav_category
    FROM (
        SELECT user_id, category,
            ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY SUM(amount) DESC) AS rn
        FROM orders
        GROUP BY user_id, category
    )
    WHERE rn = 1
),
-- 3. 用户活跃度标签
user_labeled AS (
    SELECT ub.*,
        CASE
            WHEN order_cnt >= 10 THEN '高频'
            WHEN order_cnt >= 5 THEN '中频'
            ELSE '低频'
        END AS freq_label,
        CASE
            WHEN category_cnt >= 3 THEN '多样性高'
            WHEN category_cnt >= 2 THEN '多样性中'
            ELSE '单一品类'
        END AS diversity_label
    FROM user_base ub
),
-- 4. 关联最爱品类
final AS (
    SELECT ul.*, uf.fav_category
    FROM user_labeled ul
    LEFT JOIN user_fav_category uf ON ul.user_id = uf.user_id
)
-- 5. 按频率×多样性交叉汇总
SELECT freq_label, diversity_label,
    COUNT(*) AS users,
    ROUND(AVG(total_amount), 1) AS avg_amount,
    ROUND(AVG(order_cnt), 1) AS avg_orders
FROM final
GROUP BY freq_label, diversity_label
ORDER BY freq_label DESC, diversity_label DESC"""
print(pd.read_sql(q5, conn))

print()
print("5层CTE，每层一个明确职责：")
print("  1. user_base: 用户基础指标")
print("  2. user_fav_category: 用户最爱品类")
print("  3. user_labeled: 用户打标签")
print("  4. final: 关联所有信息")
print("  5. 最终：交叉汇总")
print()
print("没有CTE的情况下写这个SQL，嵌套深度将让人生无可恋")

---

## 四、业务场景应用

### 场景1：给新同事解释复杂SQL

你的SQL有5层嵌套，新同事完全看不懂。

**解决方案**：用CTE重构。每个CTE代表一个逻辑步骤，加上清晰的注释。

### 场景2：GMV日报需要展示所有日期（包括无数据的日期）

业务数据有些日期没有记录（如节假日），但报表需要展示每一天并填0。

```sql
WITH RECURSIVE date_series AS (
    SELECT DATE('2026-01-01') AS dt
    UNION ALL
    SELECT DATE(dt, '+1 day') FROM date_series
    WHERE dt < DATE('2026-01-31')
),
daily_gmv AS (
    SELECT order_date, SUM(amount) AS gmv
    FROM orders GROUP BY order_date
)
SELECT ds.dt, COALESCE(dg.gmv, 0) AS gmv
FROM date_series ds
LEFT JOIN daily_gmv dg ON ds.dt = dg.order_date
ORDER BY ds.dt;
```

### 场景3：组织架构查询——找到某员工的所有下属

递归CTE是唯一选择。

### 场景4：数据分析师生产力提升

当你养成了"分析逻辑 = CTE结构"的习惯后：
- 写SQL的速度提升50%（先在脑里分解步骤→每个步骤一个CTE→填空）
- 调试速度提升3倍（可以单独SELECT每个CTE验证中间结果）
- 同事理解你的代码速度提升5倍（从上往下读 vs 从内往外读）

---

## 五、分级检验清单

### 初级检验（必须能独立完成）
- [ ] 能用CTE改写任意含子查询的SQL
- [ ] 能写出多个CTE链式定义
- [ ] 能解释CTE和子查询的核心区别
- [ ] 能说出CTE的基本语法

**答案提示**：
- CTE用WITH定义，子查询直接嵌入
- CTE读完上再读下，子查询从内往外读
- 基本语法：WITH cte_name AS (...) SELECT * FROM cte_name

### 中级检验（能解释给同事听）
- [ ] 能说出什么时候CTE优于子查询（3个条件）
- [ ] 能写出递归CTE的两要素
- [ ] 能区分临时表和CTE的适用场景
- [ ] 能解释CTE的物化行为对性能的影响

**答案提示**：
- 多次引用/超过2层嵌套/需要递归 → CTE
- 递归CTE = 锚定成员（起点）+ 递归成员（引用自身）+ UNION ALL
- 临时表适合大数据量和跨查询复用
- 不物化=每次引用都重新计算，物化=缓存结果但占内存

### 高级检验（能举一反三）
- [ ] 能设计复杂SQL的CTE架构（5层以上）
- [ ] 能用递归CTE处理层级数据
- [ ] 能根据数据库特性选择CTE物化策略
- [ ] 能写出生产级别的复杂分析SQL

**答案提示**：
- 每层CTE = 分析逻辑中的一个步骤
- 递归CTE处理树形结构
- PostgreSQL用AS MATERIALIZED强制作物化
- 生产级别SQL要考虑分区裁剪和索引策略

---

## 六、跨模块连接

| 方向 | 关联内容 | 说明 |
|------|---------|------|
| **← 本周二** | 窗口函数 | CTE+窗口函数=黄金组合 |
| **← 本周四** | JOIN | 复杂的JOIN逻辑可以用CTE分层 |
| **← 昨日** | 连续登录 | gap-and-island的SQL用CTE更清晰 |
| **→ 第7周** | 数仓ETL | ETL脚本大量使用CTE组织数据流 |
| **→ Python** | Pandas链式操作 | CTE = .pipe() / 中间变量赋值 |
| **扩展阅读** | 02analyst_thinking W1D4 | CTE与子查询完整知识图谱 |

### 横向连接：CTE vs Pandas 中间变量

```python
# Pandas的链式写法（对应CTE）
user_total = orders.groupby('user_id').agg(total=('amount', 'sum'))
high_value = user_total[user_total['total'] > 500]
result = high_value.sort_values('total', ascending=False)

# 对应CTE:
# WITH user_total AS (...),
# high_value AS (SELECT * FROM user_total WHERE total > 500)
# SELECT * FROM high_value ORDER BY total DESC
```

---

## 七、当日小结

> **今天最重要的一个认知**：CTE让你的SQL可读、可维护、可调试。一个超过3层嵌套子查询的SQL，用CTE改写后清晰度提升3倍。每个WITH子句 = 分析逻辑中的一个步骤。"先过滤→再聚合→再分群→最后汇总"。不只是语法糖——它让你的分析思路直接体现在SQL结构上。递归CTE是处理层级数据的唯一SQL方案。

**三条金句带回家**：
1. CTE的本质是给子查询起名字——让SQL从"从内往外读"变成"从上往下读"
2. 递归CTE = 锚定成员 + 递归成员 + UNION ALL——处理层级数据的唯一方案
3. 多次引用/超过2层嵌套/需要递归 → 用CTE；简单1层嵌套 → 用子查询

### 课后拓展

1. **练习**：用递归CTE生成斐波那契数列（前20项）

2. **思考**：如果CTE被多次引用，不同数据库的处理方式有什么不同？如何避免性能陷阱？

3. **预习**：明天要学的TopN/复购率/漏斗——三道经典SQL覆盖70%的日常工作

4. **面试准备**：背熟以下问题——"CTE和子查询的区别？""什么时候用CTE？""递归CTE的语法？""CTE和临时表的区别？"

---

*03analyst_thinking · Week 5 Day 5 · 2026-05-21*